# DT401 Week 2 Section 1: Exploring Data Types

**Module:** Data Fundamentals  
**Week:** 2 of 7  
**Section:** 1 of 4  
**Estimated Time:** 30 minutes

---

## A note before you start

This notebook is about **exploring** data types — not writing code. Every code cell in this notebook has already been written for you. Your job is to **run** each cell, **read** the output, and **think** about what it means.

You are not expected to write Python from scratch yet. That comes in DT402. Here, the code is a window into the data — you are looking through it, not building it.

---

## Learning Objectives

By the end of this notebook, you will be able to:

- Identify data types (integer, float, string, boolean) by inspecting real data
- Classify data as quantitative or qualitative, discrete or continuous, using a real dataset
- Explain why data type selection matters for storage, accuracy, and analysis
- Read and interpret Python and SQL output showing data type information

---

## Context Recap

In the Section 1 material, you explored how data can be classified: quantitative vs qualitative, discrete vs continuous, and the specific types — integer, float, string, boolean. This notebook brings those categories to life using a real dataset. You will see how Python and SQL label data types, and you will practise applying the classification framework you learnt to actual data columns.

---

## How to Use This Notebook

You will find three types of box throughout this notebook:

<div class="alert alert-block alert-warning">
<b>Reminder boxes:</b><br>
Important context about how this notebook connects to the other material you are studying or to your Module Challenge.
</div>

<div class="alert alert-block alert-info">
<b>What just happened?</b><br>
Explanations of what a code cell did and what you should take from the output.
</div>

<div class="alert alert-block alert-success">
<b>Your turn:</b><br>
Tasks for you to complete. Some involve editing a single value in a code cell. Others ask you to fill in a table in a markdown cell. Instructions will always be explicit — you will never be asked to write code from scratch.

Tasks may include a hidden solution.
<details>
    <summary style="color:black;font-weight:bold">Click here to reveal the solution</summary>

Solutions appear here.

</details>
</div>

---

## Setup

Run the cell below to load the libraries this notebook uses and connect to a temporary database. You should see a confirmation message.

In [1]:
import pandas as pd
%config SqlMagic.displaylimit = 0
%config SqlMagic.autopandas = True
%load_ext sql
%sql sqlite:///:memory:

print("Setup complete — pandas loaded and database ready.")

Connecting to 'sqlite:///%3Amemory%3A'

Setup complete — pandas loaded and database ready.


---

## Part 1: Seeing Data Types in a Real Dataset

### What does a real dataset look like?

Every dataset you encounter in your working life contains columns of different types. A patient record system holds names (text), ages (whole numbers), temperatures (decimal numbers), and consent flags (yes/no). A retail inventory holds product names (text), stock counts (whole numbers), prices (decimal numbers), and availability flags (yes/no).

The data types you have studied are not abstract categories — they are the building blocks of every table in every system you use.

In this Part, you will load a real weather dataset and observe how Python labels its columns.

```mermaid
flowchart TD
    A[Is the data numeric?] -->|Yes| B[Does it have decimal places?]
    A -->|No| C[Is it True or False?]
    B -->|Yes| D[Float — continuous]
    B -->|No| E[Integer — discrete]
    C -->|Yes| F[Boolean — binary]
    C -->|No| G[String — text / qualitative]

    style A fill:#ffffff,stroke:#000000,color:#000000,stroke-width:2px
    style B fill:#ffffff,stroke:#000000,color:#000000,stroke-width:2px
    style C fill:#ffffff,stroke:#000000,color:#000000,stroke-width:2px
    style D fill:#ffffff,stroke:#000000,color:#000000,stroke-width:1px
    style E fill:#ffffff,stroke:#000000,color:#000000,stroke-width:1px
    style F fill:#ffffff,stroke:#000000,color:#000000,stroke-width:1px
    style G fill:#ffffff,stroke:#000000,color:#000000,stroke-width:1px
```

**What this diagram shows:** A simple decision path for classifying any data column. Start at the top and follow the questions. You will use this logic throughout the notebook.

### Loading the dataset

The cell below loads using Python an hourly weather dataset for London covering January 2023 to early 2026 (source: Open-Meteo API). Run it and read the confirmation message.

In [2]:
# Load the weather dataset — skip the first 3 rows of metadata
df = pd.read_csv('../open-meteo-51.49N0.16W23m.csv', skiprows=3, nrows=26328, low_memory=False,
                 converters={'is_day ()': lambda x: bool(float(x))})
print(f"Dataset loaded: {len(df)} observations, {len(df.columns)} columns")

Dataset loaded: 26328 observations, 33 columns


### Viewing the first rows

Run the cell below to display the first five rows. Look at the values in each column — some are decimal numbers, some are whole numbers, and one (`time`) is text.

In [3]:
# Display a focused selection of columns
cols_to_show = ['time', 'temperature_2m (°C)', 'relative_humidity_2m (%)', 
                'rain (mm)', 'cloud_cover (%)', 'is_day ()']
#df['is_day ()'] = df['is_day ()']==1  # Convert to boolean for clarity
display(df[cols_to_show].head())

,time,temperature_2m (°C),relative_humidity_2m (%),rain (mm),cloud_cover (%),is_day ()
0,2023-01-01T00:00,11.2,94,0.1,100,False
1,2023-01-01T01:00,11.9,85,0.4,100,False
2,2023-01-01T02:00,11.0,80,0.0,100,False
3,2023-01-01T03:00,10.4,78,0.0,100,False
4,2023-01-01T04:00,10.2,77,0.0,100,False


<div class="alert alert-block alert-info">
<b>What just happened?</b><br>

You are looking at five hourly observations. Notice the different kinds of value:

- `temperature_2m (°C)` — values like 11.2, 11.9, 11.0. These have decimal places. This is **continuous** data.
- `rain (mm)` — values like 0.1, 0.4, 0.0. Also decimal. Also **continuous**.
- `cloud_cover (%)` — values like 100.0. Stored with a decimal point here, but conceptually a percentage — it could reasonably be an integer in another system.
- `is_day ()` — values of `False` or `True`. This is effectively a **binary** flag: daytime or not.
- `time` — values like `2023-01-01T00:00`. This looks like a date, but Python has stored it as **text**.

</div>

### What pandas calls each type

Python's data analysis library, pandas, has its own labels for data types. Run the cell below to see how it has classified each column.

In [4]:
# Show the data type pandas has assigned to each column
display(df[cols_to_show].dtypes)

time                            str
temperature_2m (°C)         float64
relative_humidity_2m (%)      int64
rain (mm)                   float64
cloud_cover (%)               int64
is_day ()                      bool
dtype: object

<div class="alert alert-block alert-info">
<b>What just happened?</b><br>

pandas uses its own type labels. Here is how they map to the terminology you already know:

| pandas label | category | What it means |
|:-------------|:-------------------|:--------------|
| `float64` | Quantitative, continuous | Decimal numbers — temperature, rainfall, prices |
| `int64` | Quantitative, discrete | Whole numbers — counts, IDs |
| `object` | Qualitative, text (string) | Text data — names, dates stored as text, categories |
| `bool` | Qualitative, binary (boolean) | True/False flags — is_day, active_account |

The `64` in `float64` and `int64` refers to how much computer memory the value uses. It does not change what the type means.
</div>

### Checking data quality and storage

Run the cell below. It shows how many non-missing values each column has, and how much memory the dataset is using.

In [5]:
# Summary information about the full dataset
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 26328 entries, 0 to 26327
Data columns (total 33 columns):
 #   Column                              Non-Null Count  Dtype  
---  ------                              --------------  -----  
 0   time                                26328 non-null  str    
 1   temperature_2m (°C)                 26328 non-null  float64
 2   relative_humidity_2m (%)            26328 non-null  int64  
 3   dew_point_2m (°C)                   26328 non-null  float64
 4   apparent_temperature (°C)           26328 non-null  float64
 5   rain (mm)                           26328 non-null  float64
 6   snowfall (cm)                       26328 non-null  float64
 7   weather_code (wmo code)             26328 non-null  int64  
 8   cloud_cover (%)                     26328 non-null  int64  
 9   vapour_pressure_deficit (kPa)       26328 non-null  float64
 10  wind_speed_10m (mp/h)               26328 non-null  float64
 11  wind_speed_100m (mp/h)              26328 non-null  

<div class="alert alert-block alert-info">
<b>What just happened?</b><br>

Two things are worth noticing here:

**Non-null counts:** The columns show 26,328 non-null values. If the number of non-null values is less than the total number of rows, it indicates missing data for those columns. This would be a data quality issue you will explore further in later weeks.

**Memory usage:** The dataset uses around 6.9 MB. Choosing the right data type affects how much storage a system needs. A column stored as `float64` when it only ever contains whole numbers wastes memory. At the scale of millions of records — an NHS patient database, a national retail system — those choices add up.
</div>

<div class="alert alert-block alert-warning">
<b>Workplace connection:</b><br>
In your organisation's databases, every column has a defined data type. When you read a report showing patient ages, stock quantities, or transaction amounts, those numbers are stored as specific types. Understanding what those types are helps you spot data quality issues, interpret calculations correctly, and communicate clearly with technical colleagues.
</div>

<div class="alert alert-block alert-info">
<b>Reinforcement Resource</b><br>

[Pandas getting started tutorials — pandas documentation (pandas, 2026)](https://pandas.pydata.org/docs/getting_started/intro_tutorials/index.html). This is a short, well-structured introduction to what pandas is and how it represents data. Read the first two pages: 'What kind of data does pandas handle?' and 'How do I read and write tabular data?'

*Read time: 8 minutes*

</div>

<div class="alert alert-block alert-success">
<b>Your turn: classifying the columns</b><br>

Using the decision diagram at the top of this Part, classify each column in the table below. Edit this markdown cell directly — click on it to edit, then fill in the empty cells.
</div>



| Column | Sample value | Quantitative or Qualitative? | If Quantitative: Discrete or Continuous? | If Qualitative: Nominal, Ordinal, or Binary? |
|:-------|:------------|:-----------------------------|:-----------------------------------------|:---------------------------------------------|
| `time` | `2023-01-01T00:00` | | | |
| `temperature_2m (°C)` | `11.2` | | | |
| `rain (mm)` | `0.1` | | | |
| `cloud_cover (%)` | `100.0` | | | |
| `is_day ()` | `True` | | | |


<details>
    <summary style="font-weight:bold">Click here to reveal the solution</summary>

| Column | Sample value | Quantitative or Qualitative? | If Quantitative: Discrete or Continuous? | If Qualitative: Nominal, Ordinal, or Binary? |
|:-------|:------------|:-----------------------------|:-----------------------------------------|:---------------------------------------------|
| `time` | `2023-01-01T00:00` | Qualitative | — | Nominal (each timestamp is a unique label) |
| `temperature_2m (°C)` | `11.2` | Quantitative | Continuous | — |
| `rain (mm)` | `0.1` | Quantitative | Continuous | — |
| `cloud_cover (%)` | `100.0` | Quantitative | Continuous | — |
| `is_day ()` | `True` | Qualitative | - | Binary (daytime = True, night = False) |

Note: `time` is stored as text (`object`) because CSV files do not have a native date type. A data analyst would convert this to a proper datetime type for analysis — a data quality step you will see in later weeks.

</details>

---

## Part 2: Numeric Types: Integers and Floats

### Why the integer / float distinction matters

We introduced two kinds of quantitative data: **discrete** (countable, whole numbers) and **continuous** (measurable, can take any value including decimals). In Python and SQL, these map directly to **integer** and **float** types.

The distinction matters for three reasons:

1. **Accuracy**: storing a price as an integer drops the pennies
2. **Valid operations**: you can count patients (integer); you can average temperatures (float); but the average of a list of names makes no sense at all
3. **Storage**: integers use less memory than floats for equivalent values

| Integer | Float |
| --- | --- |
| Whole numbers (0, 1, 200, -3...) | Decimal numbers (0, 0.1, 2.0, 2.5, -3.14) |
| Countable - discrete | Measurable - continuous |
| Exact values - no rounding | Approximate values at higher precision |
| Low memory usage for whole numbers | Higher memory usage for decimal numbers |

**What this diagram shows:** The key characteristics of each numeric type. Integers are exact and countable; floats are precise and measurable. The choice between them depends on the nature of the data.

### Worked example: continuous data in the weather dataset

Run the cell below to display the `temperature_2m (°C)` and `is_day ()` columns side by side for the first ten observations.

In [6]:
# Display temperature (continuous float) alongside is_day (discrete integer-like)
display(df[['time', 'temperature_2m (°C)', 'is_day ()']].head(10))

,time,temperature_2m (°C),is_day ()
0,2023-01-01T00:00,11.2,False
1,2023-01-01T01:00,11.9,False
2,2023-01-01T02:00,11.0,False
3,2023-01-01T03:00,10.4,False
4,2023-01-01T04:00,10.2,False
5,2023-01-01T05:00,10.1,False
6,2023-01-01T06:00,9.8,False
7,2023-01-01T07:00,9.8,False
8,2023-01-01T08:00,9.8,False
9,2023-01-01T09:00,10.1,True


<div class="alert alert-block alert-info">
<b>What just happened?</b><br>

Look at the two value columns:

- `temperature_2m (°C)` takes values like 11.2, 11.9, 11.0 — any decimal value within a range. This is **continuous** data stored as a **float**.
- `is_day ()` takes only `True` or `False` — it is either day or night. This is **binary** data. 

This is a useful lesson: the pandas type and the conceptual classification are related, but not always identical. A data analyst needs both.
</div>

### Worked example: what statistics make sense on which types

Run the cell below to see summary statistics for three numeric columns.

In [7]:
# Summary statistics for three numeric columns
display(df[['temperature_2m (°C)', 'rain (mm)', 'cloud_cover (%)']].describe())

,temperature_2m (°C),rain (mm),cloud_cover (%)
count,26328.000000,26328.000000,26328.000000
mean,11.920161,0.085817,67.410741
std,6.155168,0.370949,39.857020
min,-5.100000,0.000000,0.000000
25%,7.600000,0.000000,26.000000
50%,11.600000,0.000000,96.000000
75%,16.100000,0.000000,100.000000
max,33.500000,9.700000,100.000000


<div class="alert alert-block alert-info">
<b>What just happened?</b><br>

The `describe()` function calculates mean, minimum, maximum, and quartiles for each column. These calculations only make sense on **numeric** data.

- The average temperature across all 27,000+ observations was around 12°C — consistent with London's climate.
- The minimum temperature was -5.1°C; the maximum was much higher in summer.
- Rain has a mean near zero because it is dry most of the time, but spikes when it does rain.

You cannot run `describe()` meaningfully on a name column or a date-as-text column. Understanding data types tells you which analytical operations are valid.
</div>

### Guided modification: calculate a different column average

The cell below calculates the average temperature across the dataset. You are going to change one word to calculate the average rainfall instead.

In [10]:
# Calculate the average of a numeric column
average = df['rain (mm)'].mean()
print(f"Average: {average:.2f}")

Average: 0.09


<div class="alert alert-block alert-success">
<b>Your turn:</b><br>

In the cell above, replace `'temperature_2m (°C)'` with `'rain (mm)'` and run the cell again.

**What is the average hourly rainfall across the full dataset?**
</div>

Write your answer here: 0.09mm



<details>
    <summary style="font-weight:bold">Click here to reveal the solution</summary>

Change the cell to read:
```python
average = df['rain (mm)'].mean()
print(f"Average: {average:.2f}")
```

The average hourly rainfall is approximately 0.09 mm. We saw this previously in the summary statistics. This is very low, as most hours have no rain at all.

</details>

### SQL preview: how databases declare numeric types

In Section 4 this week, and in Workshop 1, you will write SQL queries. SQL requires you to declare the data type of each column when you create a table. Run the cell below to see how a database defines and stores integer and decimal data.

In [11]:
%%sql
CREATE TABLE IF NOT EXISTS clinic_daily (
    date        TEXT,
    patients    INTEGER,
    avg_wait_mins DECIMAL(5,1)
);

INSERT INTO clinic_daily VALUES
    ('2026-01-06', 42, 8.5),
    ('2026-01-07', 38, 11.2),
    ('2026-01-08', 51, 7.0);

SELECT * FROM clinic_daily;

Running query in 'sqlite:///%3Amemory%3A'

3 rows affected.

,date,patients,avg_wait_mins
0,2026-01-06,42,8.5
1,2026-01-07,38,11.2
2,2026-01-08,51,7.0


<div class="alert alert-block alert-info">
<b>What just happened?</b><br>

This SQL statement creates a small table representing daily clinic data. Notice:

- `patients INTEGER` — the number of patients seen each day is a whole number. SQL enforces this: you cannot store 42.5 patients.
- `avg_wait_mins DECIMAL(5,1)` — average waiting time needs one decimal place of precision. `DECIMAL(5,1)` means up to five digits total, with one after the decimal point.

This explicit type declaration is called a **schema**. Every column in a database table has a defined type, and the database will reject data that does not match. You will write schemas like this in Section 4 and Workshop 1.
</div>

<div class="alert alert-block alert-warning">
<b>Workplace connection:</b><br>
Think about numeric data in your organisation. Patient counts, transaction totals, error rates, processing times — each one is either discrete (a whole number) or continuous (a measurement with decimal precision). The choice of integer vs decimal affects how the system stores the data, what calculations it supports, and what happens when someone enters an unexpected value.
</div>

<div class="alert alert-block alert-success">
<b>Reflection: numeric types in your workplace</b><br>

In the markdown cell below, briefly answer:

1. Name one piece of data from your workplace that should be stored as an **integer**. Why must it be a whole number?
2. Name one piece of data from your workplace that should be stored as a **float or decimal**. What would be lost if it were stored as an integer instead?
</div>

*Write your response here.*

1. AAE (Accident & Emergency Attendances). It must be a whole number because each attendance represents a count by a patient.

2. RTT Performance. This must be stored as a decimal because performance is measured by precision and accuracy.

---

## Part 3: Text and Boolean Types

### Strings and booleans 

Not all data is numeric. The weekly learning introduced **qualitative** data — information that describes categories or attributes rather than quantities. Within qualitative data, two types are particularly important for data systems:

- **String** (text): names, descriptions, addresses, codes, dates stored as text. In Python: `object`. In SQL: `VARCHAR` (variable characters).
- **Boolean** (binary): data with exactly two possible states — True/False, Yes/No, Active/Inactive, 1/0. In Python: `bool`. In SQL: `BOOLEAN`.

Both map to the **qualitative** side of the classification you have seen in the main learning, though booleans sit specifically in the **binary categorical** sub-type.

```mermaid
flowchart TD
    A[Qualitative data] --> B[Text — describes or labels]
    A --> C[Categorical]
    B --> D[String / VARCHAR]
    C --> E[Ordinal — ordered categories]
    C --> F[Nominal — unordered categories]
    C --> G[Binary — two states only]
    G --> H[Boolean / BOOLEAN]

    style A fill:#ffffff,stroke:#000000,color:#000000,stroke-width:2px
    style B fill:#ffffff,stroke:#000000,color:#000000,stroke-width:2px
    style C fill:#ffffff,stroke:#000000,color:#000000,stroke-width:2px
    style D fill:#ffffff,stroke:#000000,color:#000000,stroke-width:1px
    style E fill:#ffffff,stroke:#000000,color:#000000,stroke-width:1px
    style F fill:#ffffff,stroke:#000000,color:#000000,stroke-width:1px
    style G fill:#ffffff,stroke:#000000,color:#000000,stroke-width:2px
    style H fill:#ffffff,stroke:#000000,color:#000000,stroke-width:1px
```

**What this diagram shows:** The qualitative branch of the classification tree, with the data types that represent each category.

### Worked example: text in the weather dataset

The `time` column looks like a date, but pandas read it as text. Run the cell below to see this.

In [ ]:
# Show the time column type and a few values
print(f"Data type of 'time': {df['time'].dtype}\n")
print("First five values:")
print(df['time'].head().to_string())

<div class="alert alert-block alert-info">
<b>What just happened?</b><br>

pandas has labelled `time` as `object` — its label for text strings. The CSV file stored the timestamps as formatted text (`2023-01-01T00:00`), so pandas has no way to know they are dates unless told explicitly.

This is a very common real-world data quality issue. Dates stored as text cannot be sorted chronologically, you cannot calculate the number of days between two of them, and analysis tools may not recognise them as dates at all. Converting text dates to proper datetime types is a standard data cleaning step — one you will encounter in later weeks.
</div>

### Worked example: a mixed-type DataFrame

The cell below creates a small example DataFrame representing customer account records — the kind of data you might find in a CRM system or financial database.

In [ ]:
# A small customer account dataset with all four types
customers = pd.DataFrame({
    'name':           ["Kwame Mensah",  "Siobhan O'Reilly", "Rajesh Kumar",  "Amara Okafor"],
    'age':            [34,              29,                  47,              52],
    'balance':        [1250.75,         340.00,              8922.50,         215.30],
    'active_account': [True,            True,                False,           True]
})

print("Customer records:")
display(customers)
print()
print("Column types:")
display(customers.dtypes)

<div class="alert alert-block alert-info">
<b>What just happened?</b><br>

This small table contains all four types:

| Column | pandas type | classification |
|:-------|:------------|:-------------------------|
| `name` | `object` | Qualitative — text (nominal) |
| `age` | `int64` | Quantitative — discrete (integer) |
| `balance` | `float64` | Quantitative — continuous (decimal) |
| `active_account` | `bool` | Qualitative — binary (boolean) |

This is the pattern you will see in almost every real-world dataset: a mix of types, each chosen because it reflects the nature of the data it stores.
</div>

### Worked example: filtering with a boolean column

Boolean columns are used to filter data. Run the cell below to select only active accounts.

In [ ]:
# Filter the DataFrame to rows where active_account is True
active_customers = customers[customers['active_account'] == True]
display(active_customers)

<div class="alert alert-block alert-info">
<b>What just happened?</b><br>

The boolean column `active_account` acts as a filter: only rows where the value is `True` are returned. Rajesh Kumar's account is inactive, so it is excluded.

In SQL, the same operation is written as `WHERE active_account = TRUE`. The concept is identical — the syntax differs. You will write this kind of query in Section 4 this week.
</div>

### Guided modification: filter for inactive accounts

The cell below is a copy of the filter above. Change `True` to `False` and run it to see which accounts are inactive.

In [ ]:
# TODO: Change True to False to filter for inactive accounts
inactive_customers = customers[customers['active_account'] == True]
display(inactive_customers)

<div class="alert alert-block alert-success">
<b>Expected output:</b> One row — Rajesh Kumar's inactive account.

<details>
    <summary style="color:black;font-weight:bold">Click here to reveal the solution</summary>

Change the line to:
```python
inactive_customers = customers[customers['active_account'] == False]
```

</details>
</div>

### SQL preview: VARCHAR and BOOLEAN

Run the cell below to see how SQL handles text and boolean data.

In [ ]:
%%sql
CREATE TABLE IF NOT EXISTS accounts (
    name           VARCHAR(100),
    age            INTEGER,
    balance        DECIMAL(10,2),
    active_account BOOLEAN
);

INSERT INTO accounts VALUES
    ('Kwame Mensah',    34, 1250.75, TRUE),
    ('Siobhan O''Reilly', 29, 340.00, TRUE),
    ('Rajesh Kumar',    47, 8922.50, FALSE),
    ('Amara Okafor',    52, 215.30,  TRUE);

SELECT name, balance, active_account
FROM accounts
WHERE active_account = TRUE;

<div class="alert alert-block alert-info">
<b>What just happened?</b><br>

This SQL table uses all four types and then filters using a boolean:

- `VARCHAR(100)` stores text up to 100 characters — names, addresses, codes.
- `INTEGER` stores whole numbers.
- `DECIMAL(10,2)` stores decimal numbers — here, a balance with two decimal places.
- `BOOLEAN` stores True/False — here, whether the account is active.
- `WHERE active_account = TRUE` returns only the rows where the boolean is True — the same logic you just applied in Python.

The syntax is different from Python, but the underlying concepts are identical. This is why understanding data types matters: it transfers across tools and languages.
</div>

<div class="alert alert-block alert-warning">
<b>Workplace connection — strings and booleans across sectors:</b><br>

- **NHS patient records:** patient name (`VARCHAR`), consent given (`BOOLEAN`), ward (`VARCHAR`), days admitted (`INTEGER`), temperature (`DECIMAL`).
- **Retail inventory:** product name (`VARCHAR`), in stock (`BOOLEAN`), stock count (`INTEGER`), unit price (`DECIMAL`).
- **Finance:** account holder name (`VARCHAR`), payment complete (`BOOLEAN`), transaction count (`INTEGER`), outstanding balance (`DECIMAL`).

Every system you interact with uses these four types.
</div>

---

## Part 4: Data Type Classification Challenge

### Putting it all together

You now have the full classification framework from the weekly learning, mapped to Python and SQL types. This Part asks you to apply that framework systematically to a set of columns from the weather dataset — mirroring the kind of analysis you would do when first encountering a new dataset at work.

Run the cell below to display a summary of ten columns: their pandas data type, a sample value, and the SQL type that would be used to store them.

In [ ]:
# Select ten representative columns and display a summary
challenge_cols = [
    'time',
    'temperature_2m (°C)',
    'relative_humidity_2m (%)',
    'rain (mm)',
    'snowfall (cm)',
    'cloud_cover (%)',
    'wind_speed_10m (mp/h)',
    'is_day ()',
    'sunshine_duration (s)',
    'weather_code (wmo code)'
]

SQL_equivalent = {'object': 'VARCHAR', 'float64': 'DECIMAL', 'int64': 'INTEGER', 'bool': 'BOOLEAN'}

summary = pd.DataFrame({
    'pandas dtype': df[challenge_cols].dtypes,
    'SQL type': [SQL_equivalent.get(dtype.name, 'OTHER') for dtype in df[challenge_cols].dtypes],
    'sample value': df[challenge_cols].iloc[0]
})

display(summary)

<div class="alert alert-block alert-success">
<b>Your turn — complete the classification table:</b><br>

Edit this markdown cell to complete the table. Use the decision diagram from Part 1, what you have learned across Parts 2 and 3, and your own reasoning.

</div>

| Column | Quantitative or Qualitative? | Discrete or Continuous? | Nominal / Ordinal / Binary? | SQL type |
|:-------|:-----------------------------|:------------------------|:----------------------------|:---------|
| `time` | | | | |
| `temperature_2m (°C)` | | | | |
| `relative_humidity_2m (%)` | | | | |
| `rain (mm)` | | | | |
| `snowfall (cm)` | | | | |
| `cloud_cover (%)` | | | | |
| `wind_speed_10m (mp/h)` | | | | |
| `is_day ()` | | | | |
| `sunshine_duration (s)` | | | | |
| `weather_code (wmo code)` | | | | |

<details>
    <summary style="font-weight:bold">Click here to reveal the solution</summary>

| Column | Quantitative or Qualitative? | Discrete or Continuous? | Nominal / Ordinal / Binary? | SQL type |
|:-------|:-----------------------------|:------------------------|:----------------------------|:---------|
| `time` | Qualitative | — | Nominal | VARCHAR |
| `temperature_2m (°C)` | Quantitative | Continuous | — | DECIMAL |
| `relative_humidity_2m (%)` | Quantitative | Continuous | — | INTEGER or DECIMAL |
| `rain (mm)` | Quantitative | Continuous | — | DECIMAL |
| `snowfall (cm)` | Quantitative | Continuous | — | DECIMAL |
| `cloud_cover (%)` | Quantitative | Continuous | — | INTEGER or DECIMAL |
| `wind_speed_10m (mp/h)` | Quantitative | Continuous | — | DECIMAL |
| `is_day ()` | Qualitative | — | Binary | BOOLEAN or INTEGER |
| `sunshine_duration (s)` | Quantitative | Continuous | — | DECIMAL |
| `weather_code (wmo code)` | Qualitative | — | Nominal | VARCHAR or INTEGER |

**Notes:**
- `time` is stored as text in this CSV. In a production database, it would be a `DATETIME` type.
- `relative_humidity_2m (%)` and `cloud_cover (%)` are stored as integer percentages in the pandas DataFrame and so are most naturally represented as `INTEGER` in SQL. This is the most efficient storage option. However, some systems might choose to store them as `DECIMAL` if they want to allow for fractional percentages (e.g., 75.5%) in the future. This would come at a greater storage cost. Since most columns are continuous, it may be more consistent and yield cleaner data manipulation code to store them as `DECIMAL` rather than `INTEGER`. This is a judgment call that depends on the specific use case and data quality considerations.
- `is_day ()` is stored as `float64` by pandas (0.0 or 1.0), but conceptually it is binary. Many databases store binary flags as `BOOLEAN`; some older systems use `INTEGER` (0 or 1).
- `weather_code (wmo code)` is stored as `object` (text). WMO codes are numeric codes from a fixed set — they could be stored as `INTEGER` or `VARCHAR` depending on the system, but they behave like nominal categories (you cannot calculate the average weather code).

</details>

---

## Reflection

Reflect on the following question and record your thoughts in the cell below. Ground your answer in an example from your workplace, using **approximately 100 words** in total.

Your answer here forms part of your Module Challenge Part A portfolio and is assessed against **MLO2:** *Critically reflect upon data management systems and their role in digital and technology solutions*.

Where you draw on external sources, cite them in Harvard style. This does not count towards your word count.

Once you commit your notebook, your Coach will be able to provide feedback on your response. You can develop your answer further before your final submission.

<div class="alert alert-block alert-info">

1. When might you store a price field as a text string instead of a decimal? Describe one specific problem that might arise, such as, in a calculation, a report, or a data import, and provide a brief example. 

</div>

*Write your responses here.*

1. 

References (if applicable):

---

## Summary

**You have completed Week 2 Section 1.** You can now:

- Identified data types in the London weather dataset using pandas `.head()`, `.dtypes`, and `.info()`, distinguishing `float64`, `int64`, `object`, and `bool` columns
- Distinguished integer and float types using the dataset's temperature and rainfall columns, then declared `INTEGER` and `DECIMAL` columns in a SQL table
- Identified string and boolean types in a customer accounts table, then filtered rows on a boolean condition in both Python and SQL
- Classified ten weather dataset columns as quantitative or qualitative, discrete or continuous, and assigned each an appropriate SQL type

**Key takeaways:**

- pandas dtype labels do not always match the conceptual classification exactly: `is_day` is stored as a float rather than a boolean, and percentage columns can reasonably be stored as either integer or decimal
- SQL requires every column's type to be declared explicitly in a schema; pandas infers a column's type automatically from the data it contains

**What comes next:**

- **Section 2** explores data structures: how these individual data types are organised into tables, lists, and key-value pairs, building on the DataFrame you have already used here.
- **Section 4** introduces SQL in depth, using the `INTEGER`, `DECIMAL`, `VARCHAR`, and `BOOLEAN` type declarations you have seen here as building blocks for the tables you create and query.
- **DT402** teaches you to write Python code using variables, functions, and logic, building on the familiarity with pandas you have started developing here.

**Action items:**

1. Save this notebook (Ctrl+S or Cmd+S)
2. Commit and push your work to your GitHub repository with the commit message `"Complete Week 2 Section 1 data types exploration"`
3. Make sure the repository shows your latest commit and that all files are up to date
4. Share a link to your repository with your Coach in the DT401.2 component on Aptem
5. Make sure your Coach has access to your repository so they can view your work and provide feedback
6. Open `DT401_Week_2_Section_2.ipynb` when ready to continue
